# IgG1 8-site flexibility calibration: shortest implicit-solvent MD verification

**Purpose**: After verifying the MD script logic is correct, migrate to the server to run the formal short MD.

**Setup**: AMBER ff14SB + GB-OBC2 implicit solvent. Reduced state (CYS->CYM cleaves interchain disulfide), Fab+Fc CA locked, hinge free. Run each of the 8 sites sequentially; 1 ns equilibration + 2 ns production.

**Kaggle**: Disable GPU, enable Internet. Full run takes about 30 minutes.

## Cell 1: Install + import libraries

In [ ]:
!pip install -q openmm 2>&1 | tail -2

import openmm, numpy as np, time, json, os, gc
from openmm import unit, CustomExternalForce
from openmm.app import (
    PDBFile, Modeller, ForceField, Simulation,
    HBonds, NoCutoff, PME,
)
print(f"OpenMM {openmm.__version__}")

## Cell 2: Parameters + PDB loading + disulfide diagnostics

In [ ]:
# ======== Parameters ========
PDB = "/kaggle/input/adcsim-md-8site/B220_235_disulfide_repaired.pdb"

# 8 conjugation sites (released after reducing 4 pairs of interchain disulfides)
TARGETS = [
    ("A-Cys223", "A", 223), ("B-Cys223", "B", 223),
    ("A-Cys229", "A", 229), ("B-Cys229", "B", 229),
    ("A-Cys232", "A", 232), ("B-Cys232", "B", 232),
    ("C-Cys214", "C", 214), ("D-Cys214", "D", 214),
]

K_KCAL   = 5.0                    # CA restraint strength (kcal/mol/A^2)
K_KJ     = K_KCAL * 4.184 * 100.0  # -> kJ/mol/nm^2
DT_NS    = 2e-6                   # 2 fs -> ns
EQ_NS    = 1.0                    # equilibration
PROD_NS  = 2.0                    # production

# SASA: atomic radii (nm)
RADII_NM = {"C": 0.17, "N": 0.155, "O": 0.152, "S": 0.180}
PROBE_NM = 0.14                   # 1.4 A water probe

# Fibonacci sphere (200 pt)
_i = np.arange(200)
_phi = np.arccos(1 - 2 * (_i + 0.5) / 200)
_theta = np.pi * (1 + 5 ** 0.5) * (_i + 1)
SPH = np.column_stack([
    np.sin(_phi) * np.cos(_theta),
    np.sin(_phi) * np.sin(_theta),
    np.cos(_phi),
])

# ===== Loading =====
pdb = PDBFile(PDB)
top = pdb.topology
print(f"{top.getNumResidues()} residues, {top.getNumAtoms()} atoms\n")

print("Disulfide bonds (SG-SG):")
for b in top.bonds():
    if b[0].name == "SG" and b[1].name == "SG":
        a, c = b[0].residue, b[1].residue
        print(f"  {a.chain.id}:CYS{int(a.id)} — {c.chain.id}:CYS{int(c.id)}")

## Cell 3: SASA function

In [ ]:
def sg_sasa(pos, sg_idx, nb_idx, nb_el):
    """SASA of a single SG atom. pos (nm), radii (nm), returns A^2."""
    r = RADII_NM["S"] + PROBE_NM
    pts = pos[sg_idx] + SPH * r
    occ = np.zeros(200, dtype=bool)
    for ni, el in zip(nb_idx, nb_el):
        if ni == sg_idx:
            continue
        rn = RADII_NM.get(el, 0.17)
        occ |= ((pts - pos[ni]) ** 2).sum(1) < (r + rn) ** 2
    area_nm2 = 4 * np.pi * r * r * (1.0 - occ.mean())
    return float(area_nm2 * 100.0)  # nm² -> A²

## Cell 4: Build system + run function

In [ ]:
def run_one_site(label, ch, rn):
    """Build system for a single site -> equilibration -> production -> return (sasa_mean, sasa_std, rmsf_A)"""
    print(f"\n{'='*50}")
    print(f"Site: {label}")
    print(f"{'='*50}")

    # ---- Reduced state: target site interchain SS -> CYM ----
    pdb2 = PDBFile(PDB)
    top2 = pdb2.topology
    for r in top2.residues():
        if r.name == "CYM":
            r.name = "CYS"
    cut = 0
    for b in top2.bonds():
        a0, a1 = b[0], b[1]
        if a0.name != "SG" or a1.name != "SG":
            continue
        r0, r1 = a0.residue, a1.residue
        if r0.chain.id == r1.chain.id:
            continue
        if (r0.chain.id == ch and int(r0.id) == rn) or \
           (r1.chain.id == ch and int(r1.id) == rn):
            r0.name = "CYM"
            r1.name = "CYM"
            cut += 1
            print(f"  CYS->CYM: {r0.chain.id}:CYS{int(r0.id)} — {r1.chain.id}:CYS{int(r1.id)}")
    assert cut == 1, f"Abnormal cleavage count: {cut}"

    # ---- Build system ----
    ff = ForceField("amber14-all.xml", "implicit/obc2.xml")
    mod = Modeller(top2, pdb2.positions)
    mod.addHydrogens(ff)
    print(f"  Hydrogens added: {mod.topology.getNumAtoms()} atoms")

    system = ff.createSystem(
        mod.topology, nonbondedMethod=NoCutoff, constraints=HBonds
    )

    # CA restraint: Fab(1-216)+Fc(241+) locked
    cf = CustomExternalForce("0.5*k*((x-x0)^2+(y-y0)^2+(z-z0)^2)")
    cf.addPerParticleParameter("k")
    cf.addPerParticleParameter("x0")
    cf.addPerParticleParameter("y0")
    cf.addPerParticleParameter("z0")
    n_locked = 0
    for at in mod.topology.atoms():
        if at.name != "CA":
            continue
        at_ch = at.residue.chain.id
        at_rn = int(at.residue.id)
        if at_ch in ("A", "B") and (at_rn <= 216 or at_rn >= 241):
            p = mod.positions[at.index]
            cf.addParticle(at.index, [K_KJ, float(p.x), float(p.y), float(p.z)])
            n_locked += 1
    system.addForce(cf)
    print(f"  Locked {n_locked} CA (Fab+Fc), hinge 217-240 free")

    # ---- Map atom indices ----
    sg_idx = ca_idx = None
    all_ca = []
    for at in mod.topology.atoms():
        at_ch = at.residue.chain.id
        at_rn = int(at.residue.id)
        nm = at.name
        if nm == "CA":
            all_ca.append(at.index)
        if at_ch == ch and at_rn == rn:
            if nm == "SG":
                sg_idx = at.index
            if nm == "CA":
                ca_idx = at.index
    all_ca = np.array(all_ca)
    assert sg_idx is not None, f"SG not found for {label}"
    assert ca_idx is not None, f"CA not found for {label}"

    # SG neighbors (<= 7 A)
    h_el_map = {}
    for at in mod.topology.atoms():
        if at.element and at.element.symbol != "H":
            h_el_map[at.index] = at.element.symbol
    init = np.array([[float(p.x), float(p.y), float(p.z)] for p in mod.positions])
    hk = np.array(list(h_el_map.keys()))
    d2 = ((init[hk] - init[sg_idx]) ** 2).sum(1)
    sg_nb = hk[d2 < 0.49]  # 0.7 nm^2
    sg_nb_el = [h_el_map[int(i)] for i in sg_nb]
    print(f"  SG idx={sg_idx}, CA idx={ca_idx}, neighbors={len(sg_nb)}")

    # ---- Integrator ----
    integ = openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picoseconds, 2.0 * unit.femtoseconds
    )
    integ.setRandomNumberSeed(7)
    sim = Simulation(mod.topology, system, integ)
    sim.context.setPositions(mod.positions)

    # ---- Minimize ----
    print("  Minimizing...")
    sim.minimizeEnergy(maxIterations=500)
    pe = sim.context.getState(getEnergy=True).getPotentialEnergy()
    print(f"  PE = {pe}")

    # ---- NVT equilibration 1 ns ----
    sim.context.setVelocitiesToTemperature(300 * unit.kelvin, 7)
    eq_steps = int(EQ_NS / DT_NS)  # 1 ns / 2e-6 ns = 500,000
    print(f"  NVT equilibration 1 ns ({eq_steps} steps)...")
    for i in range(4):
        sim.step(eq_steps // 4)
        print(f"    {(i + 1) * 250} ps")

    # ---- Production 2 ns (100 frames) ----
    prod_steps = int(PROD_NS / DT_NS)  # 1,000,000
    frame_step = 10000                 # 20 ps
    n_frames = prod_steps // frame_step  # 100
    print(f"  Production 2 ns, {n_frames} frames...")
    ca_frames = []
    sasa_vals = []
    t0 = time.time()
    for f in range(n_frames):
        sim.step(frame_step)
        pos = sim.context.getState(getPositions=True).getPositions(asNumpy=True)._value
        ca_frames.append(pos[all_ca].copy())
        sasa_vals.append(sg_sasa(pos, sg_idx, sg_nb, sg_nb_el))
        if (f + 1) % 50 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (f + 1) * n_frames - elapsed
            print(f"    frame {f+1}/{n_frames} ({elapsed:.0f}s, ETA {eta:.0f}s)")
    total_s = time.time() - t0
    print(f"  Production complete ({total_s:.0f}s)")

    # ---- Kabsch alignment + RMSF ----
    ca_frames = np.array(ca_frames)
    ref = ca_frames[0]
    cr = ref.mean(0)
    aligned = []
    for f in ca_frames:
        cm = f.mean(0)
        cov = (f - cm).T @ (ref - cr)
        U, _, Vt = np.linalg.svd(cov)
        d = np.sign(np.linalg.det(U @ Vt))
        R = U @ np.diag([1.0, 1.0, d]) @ Vt
        aligned.append((f - cm) @ R + cr)
    aligned = np.array(aligned)
    ci = int(np.where(all_ca == ca_idx)[0][0])
    traj_ca = aligned[:, ci, :]
    rmsf = float(
        np.sqrt(((traj_ca - traj_ca.mean(0)) ** 2).sum(1).mean())
    ) * 10.0  # nm -> A

    sasa_arr = np.array(sasa_vals)
    return sasa_arr.mean(), sasa_arr.std(), rmsf

## Cell 5: Run site by site

In [ ]:
results = {}
for label, ch, rn in TARGETS:
    sasa_m, sasa_s, rmsf = run_one_site(label, ch, rn)
    results[label] = {
        "sasa_mean": round(sasa_m, 1),
        "sasa_std": round(sasa_s, 1),
        "rmsf_A": round(rmsf, 3),
    }
    print(f"\n  >>> {label}: SASA={sasa_m:.1f}+/-{sasa_s:.1f} A^2, RMSF={rmsf:.3f} A")
    gc.collect()

## Cell 6: Results table + GNM comparison + export JSON

In [ ]:
# GNM benchmark (project phase2 output)
gnm = {
    "A-Cys223": 0.503, "B-Cys223": 0.496,
    "A-Cys229": 0.669, "B-Cys229": 0.688,
    "A-Cys232": 0.630, "B-Cys232": 0.672,
    "C-Cys214": 0.512, "D-Cys214": 0.461,
}

print(f"{'Site':<12s} {'SASA A^2':>14s} {'RMSF A':>8s} {'k=MD/GNM':>10s}")
print("-" * 48)
for lbl, d in results.items():
    k = d["rmsf_A"] / gnm[lbl]
    print(
        f"{lbl:<12s} {d['sasa_mean']:>7.1f}+/-{d['sasa_std']:>4.1f} "
        f"{d['rmsf_A']:>8.3f} {k:>10.2f}"
    )

# Export
output = {
    "method": "OpenMM AMBER ff14SB + GB-OBC2 implicit solvent",
    "system": "full IgG1 (4 chains), reduced inter-chain SS per site (CYS->CYM)",
    "restraints": "Fab(1-216)+Fc(241+) CA locked 5 kcal/mol/A^2, hinge(217-240) free",
    "equilibration": "1 ns NVT",
    "production": "2 ns, 100 frames @ 20 ps",
    "solvent": {
        "electrostatic": "Generalized Born OBC2",
        "hydrophobic": "SA surface tension (amber14 ff)",
        "viscous": "Langevin friction 1 ps^{-1}",
    },
    "sites": results,
}
with open("md_8site_flexibility.json", "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\n-> md_8site_flexibility.json saved. Download and connect to the adcsim model.")

---
## Appendix: change list for migrating to the server

After passing Kaggle verification, running the formal MD on the server only requires changing 4 parameters:

| Parameter | Kaggle verification value | Formal value | Note |
|---|---|---|---|
| `EQ_NS` | 1.0 | 2.0 | Extend equilibration to 2 ns |
| `PROD_NS` | 2.0 | 20.0 | Extend production to 20 ns |
| `N_SEED` | 1 | 5 | Average over 5 independent trajectories |
| Solvent | Implicit GB-OBC2 | Explicit TIP3P (if GPU available) | Optional upgrade |

All other logic (CYM disulfide cleavage, CA restraint, SASA, RMSF) remains unchanged.